# 🔬 Notebook 3: Reminder / Alert — Deep Dive

Now the fun part. Given a pile of scheduled reminders, **how does the
system decide what fires next**, and **how does it stay correct when
things fail**?

We walk a **bad → better → best** progression:

1. ❌ **V1** — thread-per-reminder with `sleep()`: simple, completely
   un-scalable.
2. 🙂 **V2** — polling loop over a sorted list: the "read the whole
   table every second" approach.
3. ✅ **V3** — min-heap priority queue: what most "good enough" systems
   actually use.
4. 🚀 **V4** — hashed hierarchical time wheel: what you reach for at
   Twitter/Kafka-scale, with the two off-by-one bugs that make wheels
   hard, tested rather than described.

Then the parts that only show up once there is more than one machine:

- 🔁 Retries with exponential backoff + dead-letter queue.
- 🏁 **Two replicas racing for the same reminder** — the double-fire, and
  the one-line `UPDATE … WHERE` that prevents it.
- 🪪 At-least-once + idempotent receivers (and why "exactly-once" is a lie
  even after you fix the race).
- 💥 **Recovering from downtime**: what to do with 30,000 overdue reminders,
  which ones to drop, and why firing them all is the wrong answer.
- 🧩 Sharding strategies at the very end.

## 🛠️ Setup

```bash
cd 06-system-designs/reminder-alert
uv sync
```

Then in VS Code: pick the `.venv` kernel (top-right of the notebook).
If it doesn't show up, run `Cmd+Shift+P` → **Reload Window**.

All code here uses only the Python standard library + `pydantic`. No
servers, no databases — everything runs in-process so you can step
through the ideas.


## ❌ V1: thread-per-reminder (the bad idea)

Spawn a thread, `sleep(delay)`, fire. No shared state, no infra — and
no hope of scaling past a few thousand reminders.


In [ ]:
import threading, time
from datetime import datetime, timezone, timedelta

fired = []

def v1_schedule(fire_at, message):
    def worker():
        delay = (fire_at - datetime.now(timezone.utc)).total_seconds()
        if delay > 0:
            time.sleep(delay)
        fired.append((message, datetime.now(timezone.utc)))
    threading.Thread(target=worker, daemon=True).start()

now = datetime.now(timezone.utc)
for i in range(5):
    v1_schedule(now + timedelta(milliseconds=100 * (i + 1)), f"msg #{i}")

time.sleep(0.7)
for msg, ts in fired:
    print(f"  {msg:>6s} fired at {ts.time().isoformat(timespec='milliseconds')}")


**Why it's bad:**

- One OS thread per reminder → dies at ~10k concurrent.
- All state lives in the process — restart = all reminders lost.
- No way to cancel a specific reminder (you'd need a handle per thread).
- Clock drift between machines means two replicas would fire twice.

Lesson: **durability + parallelism** must come from the storage layer,
not from threads.


## 🙂 V2: the polling loop

Put every reminder in a shared, sorted list (or SQL table indexed on
`fire_at`). A single loop wakes up every second and picks off whatever
is due.


In [ ]:
import bisect
from datetime import datetime, timezone, timedelta

class PollingScheduler:
    def __init__(self):
        # Sorted list of (fire_at, id, payload). bisect keeps it sorted.
        self._queue: list[tuple[datetime, str, str]] = []

    def schedule(self, fire_at, rid, payload):
        bisect.insort(self._queue, (fire_at, rid, payload))

    def tick(self, now=None):
        now = now or datetime.now(timezone.utc)
        fired = []
        # Pop all items whose fire_at <= now. Because the list is sorted,
        # we only look at the head — no full scan needed.
        while self._queue and self._queue[0][0] <= now:
            fired.append(self._queue.pop(0))
        return fired

sched = PollingScheduler()
t0 = datetime.now(timezone.utc)
for i in range(5):
    sched.schedule(t0 + timedelta(milliseconds=100 * (i + 1)), f"r{i}", f"msg {i}")

for _ in range(7):
    fired = sched.tick()
    for fire_at, rid, msg in fired:
        print(f"  fired {rid} ({msg})")
    time.sleep(0.1)


**Better than V1 because:**

- One loop handles all reminders; no thread explosion.
- The queue can live in a DB for durability.
- Cancellation = just remove the row.

**Still bad because:**

- `bisect.insort` is O(N) on insert for a Python list — ok for
  thousands, ugly at millions.
- A single loop is a single point of failure and a single-core bottleneck.
- In SQL-land, "poll every second" on a 1B-row table is expensive unless
  you're careful with indexes and `FOR UPDATE SKIP LOCKED`.
- **And it is outright wrong with more than one replica.** Two loops that
  `SELECT` the same due row will both fire it. `PollingScheduler` above pops
  from a list it owns exclusively, which quietly hides the hardest problem in
  the whole design. We fix that properly a few sections down.

## ✅ V3: min-heap priority queue (what most systems actually use)

A binary min-heap gives us O(log N) insert and O(log N) "pop earliest".
Python's `heapq` is already a min-heap.

We also introduce **multiple worker coroutines** sharing the heap — this
is the pattern you'll see in real-world schedulers like Quartz,
Sidekiq-cron, or a custom Redis-ZSET worker pool.


In [ ]:
import heapq, itertools
from datetime import datetime, timezone, timedelta

class HeapScheduler:
    def __init__(self):
        self._heap: list[tuple[float, int, str, str]] = []
        # A monotonically-increasing tiebreaker keeps the heap total-ordered
        # even when two reminders share the exact same fire_at.
        self._counter = itertools.count()
        # Track cancelled ids so we can skip them lazily on pop.
        self._cancelled: set[str] = set()

    def schedule(self, fire_at: datetime, rid: str, payload: str):
        heapq.heappush(
            self._heap,
            (fire_at.timestamp(), next(self._counter), rid, payload),
        )

    def cancel(self, rid: str):
        # Lazy cancellation: mark and skip on pop. Cheap, O(1).
        self._cancelled.add(rid)

    def next_due_in(self, now=None) -> float | None:
        """Seconds until the next reminder, or None if queue is empty."""
        now = now or datetime.now(timezone.utc)
        while self._heap and self._heap[0][2] in self._cancelled:
            _, _, rid, _ = heapq.heappop(self._heap)
            self._cancelled.discard(rid)
        if not self._heap:
            return None
        return max(0.0, self._heap[0][0] - now.timestamp())

    def pop_due(self, now=None):
        now = now or datetime.now(timezone.utc)
        fired = []
        while self._heap and self._heap[0][0] <= now.timestamp():
            ts, _, rid, payload = heapq.heappop(self._heap)
            if rid in self._cancelled:
                self._cancelled.discard(rid)
                continue
            fired.append((rid, payload))
        return fired


sched = HeapScheduler()
t0 = datetime.now(timezone.utc)
for i in range(5):
    sched.schedule(t0 + timedelta(milliseconds=50 * (i + 1)), f"r{i}", f"msg {i}")

sched.cancel("r2")  # cancel the 3rd one before it fires
time.sleep(0.4)
for rid, payload in sched.pop_due():
    print(f"  fired {rid}: {payload}")


**Why this scales:**

- O(log N) insert and pop, handles tens of millions of in-memory timers.
- A **`next_due_in()`** method lets the loop `sleep` exactly that long,
  eliminating busy-wait. This is the sweet spot for sub-second accuracy.
- Cancellation is lazy → O(1) on cancel, amortised O(log N) on pop.

**When to go beyond it:** if you have so many timers that even the log N
insert cost matters, or if you're adding/removing millions per second
(IoT fanout, Kafka timers), the heap's cache-unfriendly pointer chasing
starts to hurt. Enter the time wheel.


## 🚀 V4: hashed time wheel

Think of a clock with N slots. Each slot holds the bucket of reminders
due at "the current tick + slot offset".

- **Insert** = O(1): mod the delay by N, drop it in that slot.
- **Tick** = O(k): only the reminders in *this* bucket are examined.

Great when most reminders fire within a small horizon (e.g. seconds to
minutes). For longer delays we either (a) re-queue after one full wheel
rotation, or (b) layer wheels hierarchically — one for seconds, one for
minutes, one for hours. That's the "hashed hierarchical time wheel" used
in Netty, Kafka, and the Linux kernel.


In [ ]:
import math

class TimeWheel:
    """Hashed timing wheel: O(1) schedule, O(bucket) per tick.

    `cursor` is the slot we last visited. `advance()` moves it forward one slot
    and processes whatever landed there."""

    def __init__(self, slots: int = 60, tick_s: float = 1.0):
        self.slots = slots
        self.tick_s = tick_s
        self.buckets: list[list[tuple[int, str, callable]]] = [[] for _ in range(slots)]
        self.cursor = 0

    def _ticks(self, delay_s: float) -> int:
        """Whole ticks until this timer is due, rounded UP (never fire early).

        The `round(..., 9)` is not decoration: in binary floating point
        `0.15 / 0.05 == 2.9999999999999996`, and `int()`/`ceil()` on that
        schedules the timer a whole tick early. Timer bugs caused by this are
        maddening because they only bite for some delay/tick combinations."""
        return max(1, math.ceil(round(delay_s / self.tick_s, 9)))

    def schedule(self, delay_s: float, rid: str, cb):
        t = self._ticks(delay_s)
        slot = (self.cursor + t) % self.slots
        # How many times do we pass this slot BEFORE the tick we actually want?
        # Visits to any one slot are exactly `slots` ticks apart and the firing
        # tick is itself a visit, so the answer is floor((t - 1) / slots).
        # The tempting `t // slots` is off by one whenever t is a multiple of
        # `slots` — those timers then fire a full rotation late.
        rotations = (t - 1) // self.slots
        self.buckets[slot].append((rotations, rid, cb))

    def advance(self):
        """Advance one tick. Fire everything in the new bucket whose rotation
        counter is 0; decrement the rest."""
        self.cursor = (self.cursor + 1) % self.slots
        bucket = self.buckets[self.cursor]
        still_waiting = []
        for rotations, rid, cb in bucket:
            if rotations == 0:
                cb()
            else:
                still_waiting.append((rotations - 1, rid, cb))
        self.buckets[self.cursor] = still_waiting


# --- Test 1: delays that span more than one rotation still fire in order ---
wheel = TimeWheel(slots=5, tick_s=0.05)
fired_msgs: list[tuple[int, str]] = []      # (tick it fired on, id)
tick_no = 0
for i in range(8):                           # delays of 1..8 ticks, wheel holds 5
    wheel.schedule(0.05 * (i + 1), f"r{i}",
                   lambda i=i: fired_msgs.append((tick_no, f"r{i}")))

for _ in range(10):
    tick_no += 1
    wheel.advance()

assert [rid for _, rid in fired_msgs] == [f"r{i}" for i in range(8)], fired_msgs
assert [t for t, _ in fired_msgs] == [1, 2, 3, 4, 5, 6, 7, 8], fired_msgs
print("Fired:", [rid for _, rid in fired_msgs])
print("✅ every timer fired on exactly its own tick, including r4 (5 ticks =")
print("   one full rotation) and r5..r7 which wrap past the end of the wheel.")

# --- Test 2: the real trade-off is GRANULARITY, never early firing ---------
w = TimeWheel(slots=10, tick_s=0.1)
observed: list[tuple[float, float]] = []
tick_no = 0
for delay in (0.05, 0.10, 0.11, 0.25):
    w.schedule(delay, "x", lambda d=delay: observed.append((d, tick_no * 0.1)))
for _ in range(5):
    tick_no += 1
    w.advance()

print()
for wanted, got in observed:
    print(f"  wanted +{wanted:.2f}s → fired at +{got:.2f}s  (late by {got - wanted:+.2f}s)")
assert all(got >= wanted - 1e-9 for wanted, got in observed), "a wheel must never fire EARLY"
assert all(got - wanted < 0.1 + 1e-9 for wanted, got in observed), "…and never more than one tick late"
print("✅ lateness is bounded by one tick (0.10s); nothing ever fired early.")

Two things to take away from that cell.

**The `rotations` counter is what makes long delays safe.** `r4`'s 250 ms delay
is exactly one full rotation of a 5-slot, 50 ms wheel, so it lands back in the
slot the cursor is standing on. Without the counter it would fire immediately;
with `t // slots` instead of `(t - 1) // slots` it would fire a whole rotation
*late*. Both are silent, both are classic.

**The real cost of a wheel is granularity, not ordering.** A timer never fires
early, but it can fire up to one `tick_s` late — you are quantising time. Test 2
measures exactly that. So:

| | Heap (`heapq`) | Time wheel |
|---|---|---|
| Insert | O(log N) | **O(1)** |
| Cancel | O(1) lazy, O(log N) amortised | **O(1)** if you keep a slot back-pointer |
| Fire next | O(log N) | **O(1)** amortised, O(bucket) worst case |
| Accuracy | exact — sleep until the head's timestamp | **± one tick** |
| Memory locality | pointer-chasing, cache-hostile | contiguous buckets, cache-friendly |
| Long delays | free | need rotation counters or a hierarchy of wheels |

**When the wheel is the wrong answer:** if delays are wildly spread (some
seconds, some months), a single wheel wastes rotations and a hierarchy adds
cascade cost — every timer gets copied down from the hour wheel to the minute
wheel to the second wheel. And if you need millisecond accuracy you must shrink
`tick_s`, which means more ticks per second doing nothing. The heap has neither
problem; it just costs O(log N).

Most real systems run a **heap or time wheel in memory** sitting on top of a
**durable SQL store** that is the source of truth.

## 🔁 Retries + dead-letter queue

Networks fail. Push providers 5xx. We must retry — but politely, with
**exponential backoff + jitter** so we don't hammer a failing provider.

After N failed attempts, we move the reminder to a **dead-letter queue**
for a human to inspect.


In [ ]:
import random

def backoff_delay(attempt: int, base: float = 1.0, cap: float = 60.0) -> float:
    """Exponential backoff with full jitter. `attempt` starts at 1."""
    exp = min(cap, base * (2 ** (attempt - 1)))
    return random.uniform(0, exp)

random.seed(0)
for attempt in range(1, 6):
    print(f"attempt {attempt}: retry in {backoff_delay(attempt):5.2f}s")


In [ ]:
# A tiny delivery driver that retries up to MAX_ATTEMPTS, then DLQs.
from dataclasses import dataclass, field

MAX_ATTEMPTS = 3
dead_letter: list[dict] = []

@dataclass
class FakeProvider:
    fail_first_n: int = 2            # fail the first N attempts
    calls: int = 0

    def send(self, message: str) -> bool:
        self.calls += 1
        if self.calls <= self.fail_first_n:
            raise RuntimeError(f"provider 5xx (call {self.calls})")
        return True

def deliver_with_retry(provider, rid: str, message: str):
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            provider.send(message)
            print(f"  ✅ {rid} delivered on attempt {attempt}")
            return
        except Exception as e:
            print(f"  ⚠️  {rid} attempt {attempt} failed: {e}")
            if attempt == MAX_ATTEMPTS:
                dead_letter.append({"rid": rid, "msg": message, "error": str(e)})
                print(f"  💀 {rid} moved to dead-letter queue")
                return
            # In production we'd re-enqueue with `backoff_delay(attempt)` seconds.

deliver_with_retry(FakeProvider(fail_first_n=1), "r1", "Take your pills")
print()
deliver_with_retry(FakeProvider(fail_first_n=5), "r2", "Take your pills")
print(f"\nDLQ: {dead_letter}")


## 🏁 Two replicas, one reminder — where "exactly-once" actually dies

Everything so far assumed **one** scheduler. In production there are dozens, and
the moment there are two, the interesting question is not *which data structure*
but **who owns this row right now**.

Below is the smallest honest model of a SQL-backed scheduler: a table with a
`status` column, replicas that `SELECT` due rows and then act on them. Watch
what happens when two replicas read before either writes.

In [ ]:
class ReminderTable:
    """Stand-in for one SQL table. Only the operations that matter."""

    def __init__(self, rows):
        self.rows = {r["id"]: dict(r) for r in rows}
        self.selects = 0

    def due(self, now, limit=100):
        """SELECT id FROM reminders WHERE status='scheduled' AND fire_at<=:now LIMIT n"""
        self.selects += 1
        return [dict(r) for r in self.rows.values()
                if r["status"] == "scheduled" and r["fire_at"] <= now][:limit]

    def mark_firing_unsafe(self, rid, worker):
        """UPDATE reminders SET status='firing' WHERE id=:id     ← no precondition"""
        self.rows[rid].update(status="firing", owner=worker)

    def claim(self, rid, worker, now, lease_s=30) -> int:
        """UPDATE reminders SET status='firing', owner=:w, lease_until=:now+lease
           WHERE id=:id AND status='scheduled'          ← the precondition IS the lock

        Returns rows-affected. In real SQL this is a single atomic statement, so
        exactly one of two concurrent callers gets 1 and the other gets 0.
        (`SELECT … FOR UPDATE SKIP LOCKED` is the batch version of the same idea:
        it hands each replica a disjoint slice instead of making them collide.)"""
        r = self.rows[rid]
        if r["status"] != "scheduled":
            return 0
        r.update(status="firing", owner=worker, lease_until=now + lease_s)
        return 1


ROWS = [{"id": "r1", "fire_at": 0, "status": "scheduled"}]

# ---------- ❌ BAD: read-then-write. Two replicas overlap. ----------------
tbl = ReminderTable(ROWS)
batch_a = tbl.due(now=1)          # replica A reads
batch_b = tbl.due(now=1)          # replica B reads the SAME row, before A writes
fires = []
for worker, batch in (("A", batch_a), ("B", batch_b)):
    for r in batch:
        tbl.mark_firing_unsafe(r["id"], worker)
        fires.append((worker, r["id"]))
print("BAD  →", fires)
assert len(fires) == 2, "expected the classic double-fire"
print("     the user's phone buzzes twice. Nothing errored; nothing logged.")

# ---------- ✅ BEST: conditional claim. The WHERE clause decides. ---------
tbl = ReminderTable(ROWS)
batch_a = tbl.due(now=1)
batch_b = tbl.due(now=1)          # same overlapping read — that part is unavoidable
fires = []
for worker, batch in (("A", batch_a), ("B", batch_b)):
    for r in batch:
        if tbl.claim(r["id"], worker, now=1):     # only one UPDATE affects a row
            fires.append((worker, r["id"]))
print("BEST →", fires)
assert len(fires) == 1, fires
print("     the loser simply gets 0 rows affected and moves on. No coordination,")
print("     no lock service, no leader election — one atomic UPDATE.")

### …and here is why it is still **at-least-once**

The claim made *scheduling* single-owner. It did not make *delivery* exactly-once,
because the owner can die at the worst possible moment: after the push provider
accepted the message, before the row is marked `done`.

Something has to reclaim rows owned by dead workers, or a crash strands a
reminder forever. That something is the **lease**, and the lease is precisely
what guarantees you will sometimes deliver twice.

In [ ]:
def reap_expired_leases(tbl, now) -> int:
    """UPDATE reminders SET status='scheduled', owner=NULL
       WHERE status='firing' AND lease_until <= :now"""
    n = 0
    for r in tbl.rows.values():
        if r["status"] == "firing" and r.get("lease_until", 0) <= now:
            r.update(status="scheduled"); r.pop("owner", None)
            n += 1
    return n

tbl = ReminderTable(ROWS)
sent_to_provider = []

assert tbl.claim("r1", "A", now=0, lease_s=5) == 1
sent_to_provider.append("APNs accepted push (worker A)")
# …worker A's process is OOM-killed right here, before it can write status='done'…

assert reap_expired_leases(tbl, now=6) == 1          # lease expired → row is free again
assert tbl.claim("r1", "B", now=6) == 1
sent_to_provider.append("APNs accepted push (worker B)")

print("\n".join(sent_to_provider))
assert len(sent_to_provider) == 2
print("\n→ Duplicate delivered. This is not a bug you can fix; it is the price of")
print("  surviving a crash. The only lever you have is the lease length:")
print()
print(f"  {'lease':>8}  {'worst-case delay before retry':>30}  duplicate risk")
for lease in (5, 30, 300):
    print(f"  {lease:>6}s  {lease:>28}s  {'high' if lease <= 5 else 'medium' if lease <= 30 else 'low'}")
print()
print("  Short lease  → fast recovery, more duplicates (a slow worker gets reaped")
print("                 while it is still alive and working).")
print("  Long lease   → few duplicates, but a crash means the reminder is late by")
print("                 the whole lease. For a 'meeting in 5 minutes' push, a")
print("                 300s lease means the reminder is worthless when it lands.")
print()
print("  So: at-least-once on the wire + an idempotent receiver (next section).")

## 🪪 At-least-once + idempotent receivers

**Exactly-once delivery across a network is a lie** — there's always a
window where you've sent the message but haven't recorded "sent" yet,
so on crash-recovery you re-send.

The fix: accept **at-least-once** on the sender, make the receiver
**idempotent** via a stable `delivery_id`.


In [ ]:
# Simulate a crash: the worker sends twice because its "sent" record
# was lost before ack. The receiver dedupes on delivery_id.

seen_delivery_ids: set[str] = set()

def receiver(delivery_id: str, message: str) -> str:
    if delivery_id in seen_delivery_ids:
        return "duplicate — ignored"
    seen_delivery_ids.add(delivery_id)
    return f"delivered: {message}"

# Reminder r42, first attempt
print(receiver("r42-a1", "Take your pills"))
# Worker crashed before marking sent → retries with the SAME delivery id.
print(receiver("r42-a1", "Take your pills"))
# Different reminder entirely
print(receiver("r43-a1", "Meeting in 5 minutes"))


**Gotchas when picking `delivery_id`:**

- Must be deterministic per (reminder, attempt-grouping) — usually
  `f"{reminder_id}-{fire_epoch}"` so retries of the same firing collapse.
- If you bump it on every retry, you lose dedupe and the receiver sees
  duplicates.
- For recurring reminders, include the fire time in the id so Monday's
  8 AM and Tuesday's 8 AM have different ids.


## 💥 What happens to reminders during downtime?

Every scheduler is down sometimes. The interesting design question is not how to
avoid it — it is **what you do with the backlog when you come back**, and this is
the part candidates almost always skip.

Say the dispatcher is down for 45 minutes. On restart, tens of thousands of
reminders are overdue *simultaneously*. Firing them all is the obvious move and
it is wrong three times over:

1. **You melt the push provider.** APNs/FCM/Twilio rate-limit per account; a
   burst gets you throttled or temporarily banned, which extends the outage.
2. **You spam the user.** Forty notifications arriving in one second is worse
   for the user than having missed them.
3. **Some of them are now harmful.** "Take your pills 💊" delivered 3 hours late
   is not a late reminder, it is *wrong information* — the user may double-dose.
   "Your meeting starts in 5 minutes" is simply noise an hour later.

So recovery is a **policy**, and the policy is per reminder type.

In [ ]:
import math, random
from collections import defaultdict

# --- Build a realistic backlog: 45 minutes of missed fires ----------------
random.seed(11)
OUTAGE_S = 45 * 60
NOW = OUTAGE_S                      # we come back up at the end of the outage
backlog = []
for i in range(30_000):
    backlog.append({
        "id": f"r{i}",
        "user": f"u{random.randrange(5_000)}",           # 30k reminders, 5k users
        "fire_at": random.uniform(0, OUTAGE_S),          # spread over the outage
        "kind": random.choice(["medication", "meeting", "digest"]),
    })

# Per-kind staleness cutoffs. Past the cutoff, delivering is worse than dropping.
MAX_STALENESS = {"medication": 15 * 60, "meeting": 5 * 60, "digest": 24 * 3600}

def recover(backlog, now, drain_rate_per_s, coalesce=True, use_cutoffs=True):
    fresh, dropped = [], []
    for r in backlog:
        cutoff = MAX_STALENESS[r["kind"]] if use_cutoffs else math.inf
        (fresh if now - r["fire_at"] <= cutoff else dropped).append(r)

    coalesced = 0
    if coalesce:
        by_user, merged = defaultdict(list), []
        for r in fresh:
            by_user[r["user"]].append(r)
        for user, rs in by_user.items():
            if len(rs) > 1:
                coalesced += len(rs) - 1
                merged.append({"user": user, "summary": f"{len(rs)} reminders"})
            else:
                merged.append(rs[0])
        fresh = merged

    # Oldest first, drained at a fixed rate the provider is happy with.
    return {
        "delivered": len(fresh),
        "dropped": len(dropped),
        "coalesced_away": coalesced,
        "drain_seconds": max(1, math.ceil(len(fresh) / drain_rate_per_s)),
        "peak_per_s": min(len(fresh), drain_rate_per_s),
    }

PROVIDER_RATE = 2_000     # messages/s the push provider will accept from us

print(f"backlog after a {OUTAGE_S//60}-minute outage: {len(backlog):,} reminders\n")
print(f"{'policy':<44}{'delivered':>10}{'dropped':>9}{'merged':>8}{'drain':>8}{'peak/s':>10}")
for label, kw in [
    ("fire everything, as fast as we can",     dict(coalesce=False, use_cutoffs=False, drain_rate_per_s=10**9)),
    ("fire everything, rate-limited",          dict(coalesce=False, use_cutoffs=False, drain_rate_per_s=PROVIDER_RATE)),
    ("+ drop stale (per-kind cutoff)",         dict(coalesce=False, use_cutoffs=True,  drain_rate_per_s=PROVIDER_RATE)),
    ("+ coalesce per user  ← ship this",       dict(coalesce=True,  use_cutoffs=True,  drain_rate_per_s=PROVIDER_RATE)),
]:
    r = recover(backlog, NOW, **kw)
    print(f"{label:<44}{r['delivered']:>10,}{r['dropped']:>9,}{r['coalesced_away']:>8,}"
          f"{r['drain_seconds']:>7,}s{r['peak_per_s']:>10,}")

print()
print(f"Provider will accept {PROVIDER_RATE:,}/s. Look at the peak/s column: the")
print("default policy asks it for 30,000 in one second — 15x over the limit, so in")
print("reality you get throttled and the outage keeps going.")
print("Row 4 is a fraction of the volume, drains in seconds, and is *kinder* to")
print("the user — the medication reminders that are now dangerous were dropped,")
print("and the rest arrived as one 'while you were away' summary.")

### The rest of the downtime checklist

- **Never lose the intent.** The heap and the wheel are caches. Rebuild them on
  boot with `SELECT … WHERE status='scheduled' AND fire_at <= now + horizon`.
  If the outage was longer than the horizon, that same query drains the backlog
  — no special code path, which is exactly why the DB stays the source of truth.
- **Separate the catch-up lane from the live lane.** A single queue means the
  backlog starves reminders that are due *right now*. Two queues, with the live
  one getting most of the drain budget, keeps steady-state latency intact while
  the backlog trickles out.
- **Drop is a product decision, not an engineering one.** Write the cutoffs down
  as config per reminder type, and make "we dropped 8,412 medication reminders"
  a metric someone is paged about. Silently dropping is much worse than the
  outage itself.
- **Jitter the catch-up.** Reminders that were scheduled for the same `:00`
  second will otherwise re-collide the instant you resume.
- **The other kind of downtime is the provider's.** If APNs is down, your
  scheduler is fine and every delivery still fails. That is what the dead-letter
  queue above is for — plus a circuit breaker so you stop hammering it, and a
  decision about whether a reminder that DLQ'd for 2 hours is still worth sending.

## 🧩 Sharding & topology notes

At 100k fires/s peak you can't have a single scheduler. Common patterns:

- **Shard by `user_id`** — each worker owns a slice of users. Simple, but
  a whale user can overwhelm one shard.
- **Shard by `hash(fire_at)` bucket** — spread the top-of-hour thundering
  herd across many shards, even for the same user.
- **Hot ring** — keep the next ~60s of reminders in Redis (time wheel or
  ZSET), everything beyond 60s in SQL. The dispatcher pulls from SQL into
  the ring as the horizon approaches.

Whatever the topology, the SQL store remains the source of truth — a
crashed in-memory ring is rehydrated from SQL on restart.


## 🎯 Closing thoughts

- Start with **V2 polling + a conditional `UPDATE … WHERE status='scheduled'`**
  (or `SELECT … FOR UPDATE SKIP LOCKED`). It gets most real products through
  their first 10M users, and it is the piece that makes multiple replicas safe.
- Graduate to **V3 min-heap** in front of the DB when sub-second accuracy matters.
- Reach for **V4 time wheel** when insert/cancel rate is the bottleneck, not fire
  latency — and remember you are trading exactness for a bounded lateness of one tick.
- **Leases make crashes recoverable and duplicates inevitable.** Pick the lease
  length deliberately; then design receivers to be idempotent. At-least-once is
  free; exactly-once is a mirage.
- **Always** store UTC, remember the user's tz, do recurrence math in the local
  zone, and derive each occurrence from the anchor rather than the previous fire.
- **Have an answer for the outage.** Staleness cutoffs, per-user coalescing, and
  a rate-limited drain, with the live lane protected from the catch-up lane.

If you can defend those six points in an interview, you can design a
reminder/alert system. 🎉